In [6]:
import spacy
from spacy import displacy
from collections import Counter
import pandas as pd

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Sample Amazon product reviews
reviews = [
    "I absolutely love my new iPhone 15 Pro Max! The camera quality is amazing and battery life lasts all day.",
    "The Samsung Galaxy S24 Ultra is disappointing. The screen has issues and the phone overheats frequently.",
    "My Sony WH-1000XM4 headphones are fantastic! Noise cancellation works perfectly for my commute.",
    "The Dell XPS 13 laptop has terrible battery life. Would not recommend to anyone.",
    "Apple MacBook Pro with M3 chip is worth every penny. Performance is incredible for video editing.",
    "Bose QuietComfort headphones broke after just 2 months of use. Very poor durability.",
    "The Google Pixel 8 Pro takes stunning photos and the AI features are really helpful.",
    "Microsoft Surface Pro 9 is okay, but the keyboard cover is overpriced and not very durable."
]

def extract_entities_and_sentiment(text):
    """Extract entities and perform rule-based sentiment analysis"""
    doc = nlp(text)

    # Extract entities - focusing on products and brands
    entities = {
        'PRODUCT': [],
        'ORG': [],  # Brands often tagged as ORG
        'GPE': []   # Sometimes companies are tagged as GPE
    }

    for ent in doc.ents:
        if ent.label_ in entities:
            entities[ent.label_].append(ent.text)

    # Rule-based sentiment analysis
    positive_words = ['love', 'amazing', 'fantastic', 'perfectly', 'incredible',
                     'stunning', 'helpful', 'worth', 'great', 'excellent', 'good']
    negative_words = ['disappointing', 'terrible', 'poor', 'broken', 'broke',
                     'overpriced', 'issues', 'overheats', 'bad']

    positive_count = sum(1 for word in positive_words if word in text.lower())
    negative_count = sum(1 for word in negative_words if word in text.lower())

    if positive_count > negative_count:
        sentiment = "POSITIVE"
    elif negative_count > positive_count:
        sentiment = "NEGATIVE"
    else:
        sentiment = "NEUTRAL"

    sentiment_score = positive_count - negative_count

    return {
        'text': text,
        'entities': entities,
        'sentiment': sentiment,
        'sentiment_score': sentiment_score,
        'positive_words_found': [word for word in positive_words if word in text.lower()],
        'negative_words_found': [word for word in negative_words if word in text.lower()]
    }

# Process all reviews
results = []
print("AMAZON PRODUCT REVIEWS ANALYSIS")
print("=" * 50)

for i, review in enumerate(reviews, 1):
    print(f"\nReview {i}:")
    print(f"Text: {review}")

    analysis = extract_entities_and_sentiment(review)
    results.append(analysis)

    # Display results
    print(f"Entities Found:")
    for entity_type, entities_list in analysis['entities'].items():
        if entities_list:
            print(f"  {entity_type}: {', '.join(entities_list)}")

    print(f"Sentiment: {analysis['sentiment']} (Score: {analysis['sentiment_score']})")
    if analysis['positive_words_found']:
        print(f"Positive words: {', '.join(analysis['positive_words_found'])}")
    if analysis['negative_words_found']:
        print(f"Negative words: {', '.join(analysis['negative_words_found'])}")
    print("-" * 50)

# Summary statistics
print("\n" + "=" * 50)
print("SUMMARY STATISTICS")
print("=" * 50)

sentiment_counts = Counter([result['sentiment'] for result in results])
print(f"Sentiment Distribution:")
for sentiment, count in sentiment_counts.items():
    print(f"  {sentiment}: {count} reviews")

# Extract all products and brands
all_products = []
all_brands = []

for result in results:
    all_products.extend(result['entities']['PRODUCT'])
    all_products.extend(result['entities']['ORG'])
    all_products.extend(result['entities']['GPE'])

# Remove duplicates
unique_products = list(set(all_products))
print(f"\nUnique Products/Brands Mentioned: {', '.join(unique_products)}")

# Create a DataFrame for better visualization
df_results = pd.DataFrame(results)
print(f"\nDetailed Results DataFrame:")
print(df_results[['text', 'sentiment', 'sentiment_score']].to_string(index=False))

# Visualize NER for one sample review
print(f"\n" + "=" * 50)
print("NER VISUALIZATION FOR SAMPLE REVIEW")
print("=" * 50)
sample_review = reviews[0]
doc = nlp(sample_review)
print(f"Review: {sample_review}")
print("\nEntities found:")
for ent in doc.ents:
    print(f"  {ent.text}: {ent.label_}")





AMAZON PRODUCT REVIEWS ANALYSIS

Review 1:
Text: I absolutely love my new iPhone 15 Pro Max! The camera quality is amazing and battery life lasts all day.
Entities Found:
Sentiment: POSITIVE (Score: 2)
Positive words: love, amazing
--------------------------------------------------

Review 2:
Text: The Samsung Galaxy S24 Ultra is disappointing. The screen has issues and the phone overheats frequently.
Entities Found:
Sentiment: NEGATIVE (Score: -3)
Negative words: disappointing, issues, overheats
--------------------------------------------------

Review 3:
Text: My Sony WH-1000XM4 headphones are fantastic! Noise cancellation works perfectly for my commute.
Entities Found:
  ORG: Sony WH-1000XM4
Sentiment: POSITIVE (Score: 2)
Positive words: fantastic, perfectly
--------------------------------------------------

Review 4:
Text: The Dell XPS 13 laptop has terrible battery life. Would not recommend to anyone.
Entities Found:
Sentiment: NEGATIVE (Score: -1)
Negative words: terrible
-----